In [1]:
from neo4j import GraphDatabase
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm import OpenAILLM, OllamaLLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.generation.prompts import ERExtractionTemplate
from dotenv import load_dotenv
import pandas as pd
from langchain_core.documents import Document
from langchain_community.llms import Ollama
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever, Text2CypherRetriever
from neo4j_graphrag.schema import get_schema
from neo4j_graphrag.generation import GraphRAG


import os, time, asyncio, glob, csv

d:\GitHub\muict_thai_legal_RAG_GraphRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Initialize Graph Database 

In [2]:
load_dotenv(dotenv_path="../.env")

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE_NAME = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

### 1.1 Call BGE-M3 Embedding model

In [3]:
import torch
from FlagEmbedding import BGEM3FlagModel

# เช็คว่ามี GPU (CUDA) หรือไม่
# ถ้ามีให้ใช้ 'cuda' ถ้าไม่มีให้ใช้ 'cpu'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_fp16 = True if device == 'cuda' else False  # fp16 ส่วนใหญ่รองรับเฉพาะบน GPU

print(f"กำลังรันโมเดลบน: {device.upper()}")

# 2. Initialize Model
embedder = BGEM3FlagModel(
    'VISAI-AI/nitibench-ccl-human-finetuned-bge-m3', 
    use_fp16=use_fp16, 
    device=device 
)

กำลังรันโมเดลบน: CUDA


Fetching 23 files: 100%|██████████| 23/23 [00:00<?, ?it/s]


### 1.2 Wrap embedding model
เนื่องจาก Embedding model ที่จะใช้ใน Neo4J (เรียกจาก library Neo4J โดยตรง ไม่ใช่ langchain) ไม่สามารถเรียกใช้ Embedding model BGE-M3 ได้โดยตรง ตัว Library รองรับแค่พวก Commercial Models e.g. openai จึงต้องมีการ Wrap เพื่อให้ใช้งานได้ 

In [4]:
from neo4j_graphrag.embeddings import Embedder

class Neo4jCompatibleEmbedder(Embedder):
    def __init__(self, original_embedder):
        self.original_embedder = original_embedder

    def embed_query(self, text: str) -> list[float]:
        # 1. เรียกโมเดล M3 (สมมติว่าใช้ .encode)
        result = self.original_embedder.encode(text)
        
        # 2. ตรวจสอบว่าผลลัพธ์เป็น dict หรือไม่ (BGE-M3 มักคืนค่าแบบนี้ถ้าไม่ได้ตั้ง return_dense_only=True)
        if isinstance(result, dict):
            # ดึงเฉพาะ Dense Vector ออกมา
            embedding = result.get('dense_vecs')
        else:
            embedding = result

        # 3. แปลงจาก numpy array เป็น list เพื่อให้ Pydantic/Neo4j ยอมรับ
        if hasattr(embedding, "tolist"):
            return embedding.tolist()
        
        # กรณีเป็น list อยู่แล้วแต่สมาชิกข้างในอาจเป็น numpy types ให้วนลูปแปลง (ถ้าจำเป็น)
        return [float(x) for x in embedding]

    def embed_nodes(self, nodes):
        for node in nodes:
            # node.text คือเนื้อหาที่ถูก Chunk แล้ว
            node.embedding = self.embed_query(node.text)

# การใช้งาน
# ระวังอย่ารันบรรทัดนี้ซ้ำ (เพราะจะกลายเป็น wrap ซ้อน wrap)
neo4j_embedder = Neo4jCompatibleEmbedder(embedder)

In [5]:
df = pd.read_parquet('../data/processed/ref_vectors_sparse_nitibench-ccl-bge-m3.parquet')
df.head()

,id,law_name,section_num,text,dense_vector,sparse_vector,reference_laws
0,0,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546,132,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มา...,"[-0.03570556640625, 0.004070281982421001, -0.0...","{""221767"": 0.060760498046875, ""70390"": 0.15673...",[{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน...
1,1,ประมวลกฎหมายแพ่งและพาณิชย์,1598/5,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู...,"[0.0345458984375, 0.049072265625, -0.020095825...","{""130047"": 0.0031299591064453125, ""57467"": 0.0...",[]
2,2,ประมวลกฎหมายแพ่งและพาณิชย์,876,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 876\nถ้าผู้รั...,"[-0.039886474609375, 0.05941772460937501, 0.00...","{""167120"": 0.015411376953125, ""382"": 0.0497741...",[]
3,3,ประมวลกฎหมายแพ่งและพาณิชย์,1030,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1030\nถ้าผู้เ...,"[-0.029144287109375003, 0.024978637695312, -0....","{""130047"": 0.11090087890625, ""57467"": 0.025955...",[]
4,4,ประมวลกฎหมายแพ่งและพาณิชย์,849,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 849\nการรับเง...,"[0.011123657226562, 0.0675048828125, -0.041503...","{""130047"": 0.00732421875, ""57467"": 0.020690917...",[]


In [ ]:
len(df)

3833

In [ ]:
df['reference_laws'] = df['reference_laws'].astype(str)

In [8]:
df['law_name'].value_counts()

law_name
ประมวลกฎหมายแพ่งและพาณิชย์                                                                                          1661
พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535                                                                   398
ประมวลรัษฎากร                                                                                                        351
พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535                                                                              199
พระราชบัญญัติธุรกิจสถาบันการเงิน พ.ศ. 2551                                                                           165
พระราชบัญญัติการประกอบกิจการพลังงาน พ.ศ. 2550                                                                        153
พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546                                                                          131
พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514                                                                          99
พระราชกำหนดการประกอบธุร

In [ ]:
df[df['law_name'] == 'ประมวลกฎหมายแพ่งและพาณิชย์'].head()

,id,law_name,section_num,text,dense_vector,sparse_vector,reference_laws
1,1,ประมวลกฎหมายแพ่งและพาณิชย์,1598/5,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู...,"[0.0345458984375, 0.049072265625, -0.020095825...","{""130047"": 0.0031299591064453125, ""57467"": 0.0...",[]
2,2,ประมวลกฎหมายแพ่งและพาณิชย์,876,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 876\nถ้าผู้รั...,"[-0.039886474609375, 0.05941772460937501, 0.00...","{""167120"": 0.015411376953125, ""382"": 0.0497741...",[]
3,3,ประมวลกฎหมายแพ่งและพาณิชย์,1030,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1030\nถ้าผู้เ...,"[-0.029144287109375003, 0.024978637695312, -0....","{""130047"": 0.11090087890625, ""57467"": 0.025955...",[]
4,4,ประมวลกฎหมายแพ่งและพาณิชย์,849,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 849\nการรับเง...,"[0.011123657226562, 0.0675048828125, -0.041503...","{""130047"": 0.00732421875, ""57467"": 0.020690917...",[]
6,6,ประมวลกฎหมายแพ่งและพาณิชย์,565,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 565\nการเช่าถ...,"[-0.03216552734375, 0.05874633789062501, -0.03...","{""57467"": 0.0092620849609375, ""167120"": 0.0239...",[]


## 2. Chunking to documents

In [48]:
START = 100
END = 150

def batch_document_append(df,start=0, end=len(df)):
    documents = []
    for index, row in df.iloc[start:end,:].iterrows():
        # สร้างเนื้อหาที่มีบริบทชัดเจนในตัวเอง
        content = f"กฎหมาย: {row['law_name']}\nเนื้อหา: {row['section_content']}"
        
        # เก็บ Metadata เพื่อใช้ในภายหลัง
        metadata = {
            "law_name": row['law_name'],
        }
        documents.append(Document(page_content=content, metadata=metadata))
    return documents

documents = batch_document_append(df,start=START, end=END)
documents

[Document(metadata={'law_name': 'พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535'}, page_content='กฎหมาย: พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535\nเนื้อหา: พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535 มาตรา 189 บุคคลใดประสงค์จะนำหลักทรัพย์ที่ตนออกไปซื้อขายในตลาดหลักทรัพย์ จะต้องนำหลักทรัพย์นั้นไปจดทะเบียนกับตลาดหลักทรัพย์\nเมื่อตลาดหลักทรัพย์ได้รับคำขอจดทะเบียนแล้ว ให้พิจารณาและเสนอความเห็นต่อคณะกรรมการตลาดหลักทรัพย์เพื่อสั่งรับหรือไม่รับเป็นหลักทรัพย์จดทะเบียน'),
 Document(metadata={'law_name': 'พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535'}, page_content='กฎหมาย: พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535\nเนื้อหา: พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535 มาตรา 205 บริษัทใดไม่ปฏิบัติตามมาตรา 109 มีความผิดทางพินัยต้องชำระค่าปรับเป็นพินัยไม่เกินสองแสนบาท และชำระค่าปรับเป็นพินัยรายวันอีกวันละสองพันบาท*จนกว่าจะปฏิบัติถูกต้อง'),
 Document(metadata={'law_name': 'พระราชบัญญัติทรัสต์เพื่อธุรกรรมในตลาดทุน พ.ศ. 2550'}, page_content='กฎหมาย: พระราชบัญญัติทรัสต์เพื่อธุรกรรมในตลาดทุน พ

## 3. Select LLM

In [5]:
from neo4j_graphrag.llm import OllamaLLM
# llm = OllamaLLM(
#     model_name="llama3",
#     # model_params={"options": {"temperature": 0}, "format": "json"},
#     host="http://localhost:11434",  # when using a remote server
# )

os.environ["OPENAI_API_KEY"] = os.getenv("openrouter_API_key") # ใส่ OpenRouter Key ของคุณตรงนี้

# 2. เรียกใช้งาน โดยใส่ base_url กำกับไว้ข้างนอก
llm = OpenAILLM(
    model_name="openai/gpt-4o-mini", 
    base_url="https://openrouter.ai/api/v1",
    model_params={
        "temperature": 0
    }
)
# llm.invoke("say hello-world")

## 4. Graph Construction

In [ ]:
# llm = ChatOpenAI(api_key=os.getenv("openrouter_API_key") ,temperature=0, model="openai/gpt-4o-mini", base_url="https://openrouter.ai/api/v1")
# llm = OllamaLLM(model="llama3", )

entities = [
    {"label": "law_name", "properties": [{"name": "law_name", "type": "STRING", "name": "section_num", "type": "STRING"}]},  
]

relations = [
    {"label": "CONTAINS", "source": "law_name", "target": "section_num"},
]

kg_builder = SimpleKGPipeline(
    driver=driver,
    llm=llm,
    embedder=neo4j_embedder,
    entities=entities,
    relations=relations,
    neo4j_database=DATABASE_NAME,
    from_pdf=False,
    )

In [50]:
import time
from langchain_community.callbacks import get_openai_callback

start_time = time.perf_counter()
with get_openai_callback() as cb:
    for i in range(len(documents)):
        #นำเข้า Neo4j ทีละ Document (ซึ่งแต่ละ Document ก็คือ Section ที่ถูก Chunk แล้ว) สิ่งที่นำเข้าเป็น json format
        await kg_builder.run_async(text=documents[i].page_content, document_metadata=documents[i].metadata,) 
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.4f} seconds")
print(f"Total Tokens: {cb.total_tokens}")
print(f"Prompt Tokens: {cb.prompt_tokens}")
print(f"Completion Tokens: {cb.completion_tokens}")

LLM response has improper format for chunk_index=0


Elapsed time: 497.3945 seconds
Total Tokens: 0
Prompt Tokens: 0
Completion Tokens: 0


## 5. Retrieval

### 5.1 Vector Retriever เหมือน RAG ธรรมดา

In [6]:
# Initialize the retriever
retriever = VectorRetriever(
     driver,
     index_name= "text_embeddings",
     embedder=neo4j_embedder,
     return_properties=["text"]
)
query = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
result = retriever.search(query_text=query, top_k=5)
result

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


RetrieverResult(items=[RetrieverResultItem(content="{'text': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'}", metadata={'score': 0.9034852981567383, 'nodeLabels': ['Section'], 'id': '4:479141a0-c13e-4e91-8882-94f287492b94:0'}), RetrieverResultItem(content="{'text': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 125 ผู้ใดประกอบกิจการในลักษณะเป็นการประกอบธุรกิจสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 16 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'}", metadata={'score': 0.8719487190246582, 'nodeLabels': ['Section'], 'id': '4:479141a0-c13e-4e91-8882-94f287492b94:2339'}), RetrieverResultItem(content="{'text': 'พระราชบัญญัติสัญญาซื

In [ ]:
# ดึงเนื้อหาของอันแรก
first_result = result.items[0].content
# ดึงค่า Score (ความเหมือน)
first_score = result.items[0].metadata['score']

print(f"Score: {first_score}")
print(f"Content: {first_result}")

Score: 0.9035258293151855
Content: {'text': 'กฎหมาย: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546\nมาตรา: 132\nเนื้อหา: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'}


### 5.2 Vector Cypher Retriever 
- ใช้ Cosine similarity เทียบความคล้าย
- Cypher query เข้าไป (ต้องมีการกำหนดเอง ดีที่สุดคืออิงจาก Schema) 

In [5]:
schema = get_schema(driver)
schema

'Node properties:\nSection {section_num: STRING, id: INTEGER, text: STRING, embedding: LIST, law_id: STRING, law_name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Section)-[:REFERENCES_TO]->(:Section)'

In [7]:
chunk_to_law_query = """
// 1. กระโดดจาก Node หลัก (Parent) ไปหา Node ที่ถูกอ้างอิง (Children)
OPTIONAL MATCH (node)-[:REFERENCES_TO]->(referencedSection)

// 2. รวบรวมข้อมูล Children ไว้เป็น List โดยดึง Text มาด้วย
WITH node, score, 
     collect({
         law_name: referencedSection.law_name, 
         section_num: referencedSection.section_num,
         text: referencedSection.text
     }) AS children_nodes

// 3. RETURN ผลลัพธ์: 
// ให้ node.text เป็น "Parent Content"
// และ Metadata เก็บข้อมูลรายละเอียดของ Parent พร้อมลิสต์ของ Children
RETURN 
    node.text AS content, 
    score, 
    {
        parent_law_name: node.law_name,
        parent_section_num: node.section_num,
        children: [c in children_nodes WHERE c.law_name IS NOT NULL]
    } AS metadata
"""

# ประกาศ retriever เหมือนเดิม
vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name="text_embeddings",
    embedder=neo4j_embedder,
    retrieval_query=chunk_to_law_query
)

In [35]:
query = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร" # ข้อ 1
result = vector_cypher_retriever.get_search_results(query_text=query, top_k=3)
print(len(result.records))
result.records[0].data()
# for item in result.items:
#     print(item.content)

3


{'content': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน',
 'score': 0.9034852981567383,
 'metadata': {'parent_section_num': '132',
  'parent_law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546',
  'children': [{'text': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 54 ศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าต้องเป็นนิติบุคคลประเภทบริษัทมหาชนจำกัดและได้รับใบอนุญาตจากคณะกรรมการ ก.ล.ต.\nศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าที่เปิดให้เฉพาะผู้ลงทุนสถาบันซื้อขายสัญญาซื้อขายล่วงหน้าเพื่อตนเองต้องเป็นนิติบุคคลประเภทบริษัทจำกัดหรือบริษัทมหาชนจำกัดและจดทะเบียนกับสำนักงาน ก.ล.ต.\nการขออนุญาต การขอจดทะเบียน การออกใบอนุญาต และการรับจดทะเบียนให้เป็นไปตามหลักเกณฑ์ ที่คณะกรรมการ ก.ล.ต. ประกาศกำหนด',
    'section_num': '54',
    'law_name': 'พระราชบัญญัติสัญญ

## 6. ใช้ Typhoon QA

In [10]:
os.environ["OPENAI_API_KEY"] = os.getenv("thai_llm_API_key")

model = "typhoon"
# 2. สร้าง LLM Object ให้ถูกต้องสำหรับ neo4j-graphrag
llm = OpenAILLM(
    model_name="typhoon-v2.5-30b-a3b-instruct",
    # สำคัญ: base_url ต้องหยุดอยู่ที่ /v1 (ห้ามมี /chat/completions ต่อท้าย)
    base_url=f"https://api.opentyphoon.ai/v1",
    # ต้องระบุ api_key เป็น string เสมอ (แม้จะเป็นค่าว่างก็ต้องใส่ "") เพื่อป้องกัน TypeError
    model_params={
        "temperature": 0.1,
        "max_tokens": 8192, 
    }
)
# llm.invoke("สวัสดี")

### คำตอบข้อ 1

In [11]:
query = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร" # ข้อ 1

result = GraphRAG(llm=llm,retriever=vector_cypher_retriever)
print(result.search(query_text=query, retriever_config= {"top_k": 10}).answer)

ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน


### คำตอบข้อ Ref เยอะๆ

In [12]:
# query = "บทบัญญัติในหมวด 2 ว่าด้วยตั๋วแลกเงินที่สามารถนำมาใช้กับตั๋วสัญญาใช้เงินมีมาตราใดบ้าง"

# result = GraphRAG(llm=llm,retriever=vector_cypher_retriever)
# print(result.search(query_text=query, retriever_config= {"top_k": 10}).answer)